In [ ]:
import io
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

BUCKET = "nbu-rates-sema-lab2"
s3 = boto3.client("s3")

frames = {}
for cur in ["usd", "eur"]:
    obj = s3.get_object(Bucket=BUCKET, Key=f"data/nbu_{cur}_2022.csv")
    frames[cur.upper()] = pd.read_csv(io.BytesIO(obj["Body"].read()), parse_dates=["date"])
    print(cur.upper(), "-", len(frames[cur.upper()]), "рядків")

frames["USD"].head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(frames["USD"]["date"], frames["USD"]["rate"], label="Долар США (USD)")
ax.plot(frames["EUR"]["date"], frames["EUR"]["rate"], label="Євро (EUR)")
ax.set_title("Офіційний курс гривні до долара США та євро у 2022 році (дані НБУ)")
ax.set_xlabel("Дата")
ax.set_ylabel("Курс, грн за 1 одиницю валюти")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
fig.autofmt_xdate()
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig("nbu_rates_2022.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
s3.upload_file("nbu_rates_2022.png", BUCKET, "plots/nbu_rates_2022.png")

for item in s3.list_objects_v2(Bucket=BUCKET)["Contents"]:
    print(item["Key"], "-", item["Size"], "байт")